# Register an MLTable Data Asset

Create MLTable metadata around a taxi CSV, register an immutable Azure ML data asset, and retrieve it for verification.

**Sources:** Adapted from [AzureML-deep-dive-L200](https://github.com/jomedinagomez/AzureML-deep-dive-L200/blob/efa77413408a8053781096c41fb07cb882c8f518/taxi-fare-predictions/notebooks/dataset_registration.ipynb) and [Azure/azureml-examples](https://github.com/Azure/azureml-examples/tree/main/sdk/python/assets/data). Both sources use the MIT License; details are in [source attribution](../../../workshop/SOURCES.md).

In [ ]:
from pathlib import Path
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential
from dotenv import dotenv_values
import shutil
import mltable
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = Path(notebook_file).resolve().parent if notebook_file else Path.cwd().resolve()
for parent in (search_start, *search_start.parents):
    candidate = parent if parent.name == "workspace_readiness" else parent / "notebooks/workspace_readiness"
    if (candidate / ".env.example").is_file():
        SUITE_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run from this repository or notebooks/workspace_readiness.")

SAMPLE_ROOT = SUITE_ROOT.parents[1] / "workshop"

env_path = SUITE_ROOT / ".env"
if not env_path.is_file():
    raise FileNotFoundError(f"Configure {env_path} with your four Azure resource values.")
env = dotenv_values(env_path, encoding="utf-8-sig", interpolate=False)
SUBSCRIPTION_ID = (env["AZURE_SUBSCRIPTION_ID"] or "").strip()
RESOURCE_GROUP = (env["AZURE_RESOURCE_GROUP"] or "").strip()
WORKSPACE_NAME = (env["AZUREML_WORKSPACE_NAME"] or "").strip()
REGISTRY_NAME = (env["AZUREML_REGISTRY_NAME"] or "").strip()

if len(env) != 4 or not all((SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME)):
    raise ValueError("Fill in only the four workspace and registry values in .env.")
if any(value.startswith(("<", "${")) for value in (SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME)):
    raise ValueError("Replace the .env placeholders with your Azure resource values.")

credential = AzureCliCredential(process_timeout=60)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
print(f"Workspace: {WORKSPACE_NAME}; registry: {REGISTRY_NAME}")

ASSET_NAMES = {
    "data": "readiness-taxi-data",
    "environment": "readiness-taxi-environment",
    "model": "readiness-taxi-model",
}



DATA_NAME = ASSET_NAMES["data"]


In [ ]:
source_csv = SAMPLE_ROOT / "data/taxi/raw/yellowTaxiData.csv"
generated_mltable = SUITE_ROOT / "outputs/foundation_mltable"
if generated_mltable.exists():
    shutil.rmtree(generated_mltable)

table = mltable.from_delimited_files(paths=[{"file": str(source_csv)}])
table.save(str(generated_mltable), colocated=True)

data_definition = Data(
    name=DATA_NAME,
    type=AssetTypes.MLTABLE,
    path=str(generated_mltable),
    description="Taxi data registered during the Azure ML workshop",
    tags={"validation": "workspace-readiness", "source": "taxi-sample"},
)

registered_data = ml_client.data.create_or_update(data_definition)
verified_data = ml_client.data.get(
    DATA_NAME,
    version=registered_data.version,
)
assert verified_data.name == DATA_NAME
assert verified_data.version == registered_data.version
print(f"Registered data: {verified_data.name}:{verified_data.version}")


## Expected Result

A local MLTable folder is prepared. On execution, Azure ML assigns an immutable data version and the notebook retrieves it successfully.

Next: `03_register_environment.ipynb`.